# CPG-RL 地形 v3b（terrain3 修正版：爬坡力回復）

診斷發現 terrain3 在 15° 爬坡「卡住磨蹭」（見 `2026-07-21-terrain3-vs-terrain2_1` 與診斷）：
根因是 `-0.5·slip` 滑動懲罰把步態學得過度保守、全地形變慢，**inference 調不了、只能重訓**。

v3b 兩處改動（其餘 terrain3 設定全留：摩擦 [0.5,1.25]、抬腳 25cm、斜坡30°/障礙12cm）：
- **滑動懲罰 `W_SLIP` 0.5 → 0.1**（元凶大幅減弱；摩擦下限拉高本身已解決物理打滑）。
- **新增爬坡/前進獎勵 `+0.6·clip(blin[0],0,cmd[0])`**（不飽和，逼策略在坡上持續用力前進、別停）。
- 順手：metrics 加 `nan_to_num` 防護（修 terrain3 監控時 slip/relh 偶爾 nan）。

> 以 `%%writefile` 寫出模組（cpg3 / terrain3 / obs3 不變，env 改名 go2_terrain3b_env）後 import，Colab 內自成一體，不動 repo 舊檔。
> ⚠️ 開訓前務必先跑 Smoke test cell。

## 第 1 步：GPU + 安裝
執行階段 → 變更執行階段類型 → GPU。`MUJOCO_GL=egl` 必須在 import mujoco 前設好。

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
!pip install -q mujoco mujoco-mjx brax mediapy
print("done")

In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())   # 要看到 cuda

## 第 1.5 步：Go2 模型（scene_mjx.xml）
clone menagerie；env 內用 `apply_pd` 把位置伺服改成等效 kp=90 / kd=3。

In [ ]:
import os, subprocess
if not os.path.exists("mujoco_menagerie"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/google-deepmind/mujoco_menagerie.git"], check=True)
SCENE = "mujoco_menagerie/unitree_go2/scene_mjx.xml"
print("model exists:", os.path.exists(SCENE))

## 第 2 步：嵌入已驗證模組（%%writefile → import）

以下四個 cell 各用 `%%writefile` 把**已本機測試**的模組原始碼原封寫成檔案，再由下一個 cell import。
順序：**cpg3 → terrain3 → obs3 → go2_terrain3b_env**（env 依賴前三者）。
每個 cell 頂端的來源註解標明對應模組，改動請同步回模組。

In [ ]:
%%writefile cpg3.py
# === 模組: cpg3.py（v3：改自 v2，Colab 內自成一體）===
"""CPG v3：論文 CPG + 每腿可學抬腳 gc；相容 fixed(12)/learnable(16) 兩種動作。"""
import numpy as np
import mujoco
import jax.numpy as jnp

MU_MIN, MU_MAX = 1.0, 2.0
OMEGA_MIN, OMEGA_MAX = 0.0, 4.5
A_CONV = 50.0
D_STEP = 0.12
G_C = 0.08                       # 固定抬腳（舊模型用）
G_P = 0.01
GC_MIN, GC_MAX = 0.05, 0.25      # 可學抬腳範圍（v3：上限放寬到 25cm 跨 12cm 障礙）
W_COUP = 8.0
N_CPG_SUB = 4
LEGS = ["FL", "FR", "RL", "RR"]
HOME3 = jnp.array([0.0, 0.9, -1.8])
HOME3_np = np.array([0.0, 0.9, -1.8])
PHASE_OFFSET = jnp.array([0.0, jnp.pi, jnp.pi, 0.0])
PHI = PHASE_OFFSET[None, :] - PHASE_OFFSET[:, None]


def detect_mode(act_dim):
    if act_dim == 12:
        return "fixed"
    if act_dim == 16:
        return "learnable"
    raise ValueError(f"未知動作維度 {act_dim}（僅支援 12/16）")


def action_to_cpg_cmd(action, mode):
    if mode == "fixed":
        a = jnp.tanh(action).reshape(4, 3)
        gc = jnp.full(4, G_C)
    else:
        a4 = jnp.tanh(action).reshape(4, 4)
        a = a4[:, :3]
        gc = (a4[:, 3] + 1) / 2 * (GC_MAX - GC_MIN) + GC_MIN
    mux = (a[:, 0] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    muy = (a[:, 1] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    omega = (a[:, 2] + 1) / 2 * (OMEGA_MAX - OMEGA_MIN) + OMEGA_MIN
    return mux, muy, omega, gc


def cpg_init():
    return {"rx": jnp.full(4, 1.5), "rx_d": jnp.zeros(4),
            "ry": jnp.full(4, 1.5), "ry_d": jnp.zeros(4), "theta": PHASE_OFFSET}


def cpg_step(c, mux, muy, omega, dt):
    rx, rxd, ry, ryd, th = c["rx"], c["rx_d"], c["ry"], c["ry_d"], c["theta"]
    h = dt / N_CPG_SUB
    for _ in range(N_CPG_SUB):
        rxd = rxd + A_CONV * (A_CONV / 4.0 * (mux - rx) - rxd) * h
        rx = rx + rxd * h
        ryd = ryd + A_CONV * (A_CONV / 4.0 * (muy - ry) - ryd) * h
        ry = ry + ryd * h
        rbar = 0.5 * (rx + ry)
        diff = th[None, :] - th[:, None] - PHI
        coup = jnp.sum(rbar[None, :] * jnp.sin(diff), axis=1)
        th = th + (2.0 * jnp.pi * omega + W_COUP * coup) * h
    th = jnp.mod(th, 2.0 * jnp.pi)
    return {"rx": rx, "rx_d": rxd, "ry": ry, "ry_d": ryd, "theta": th}


def cpg_foot_offsets(c, gc):
    th = c["theta"]
    fx = 2 * (c["rx"] - MU_MIN) / (MU_MAX - MU_MIN) - 1.0
    fy = 2 * (c["ry"] - MU_MIN) / (MU_MAX - MU_MIN) - 1.0
    dx = -D_STEP * fx * jnp.cos(th)
    dy = D_STEP * fy * jnp.cos(th)
    dz = jnp.where(jnp.sin(th) > 0, gc * jnp.sin(th), G_P * jnp.sin(th))
    return jnp.stack([dx, dy, dz], axis=-1)


def cpg_to_joint_targets(c, jinvs, gc):
    off = cpg_foot_offsets(c, gc)
    dq = jnp.einsum("kij,kj->ki", jinvs, off)
    q = HOME3[None, :] + dq
    return q.reshape(12)


def leg_ik_consts(scene_path):
    m = mujoco.MjModel.from_xml_path(scene_path); d = mujoco.MjData(m)
    jinvs = []
    for k, leg in enumerate(LEGS):
        jb = 7 + 3 * k
        gid = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, leg)
        hip = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, leg + "_hip")

        def foot(q3):
            mujoco.mj_resetDataKeyframe(m, d, 0)
            d.qpos[jb:jb + 3] = q3; mujoco.mj_forward(m, d)
            return (d.geom_xpos[gid] - d.xpos[hip]).copy()
        e = 1e-3; J = np.zeros((3, 3))
        for j in range(3):
            dq = np.zeros(3); dq[j] = e
            J[:, j] = (foot(HOME3_np + dq) - foot(HOME3_np - dq)) / (2 * e)
        jinvs.append(np.linalg.inv(J))
    return np.array(jinvs, np.float32)


In [ ]:
%%writefile terrain3.py
# === 模組: terrain3.py（v3：改自 v2，Colab 內自成一體）===
"""地形 v3：統一 hfield（平台 + 0–30° 斜坡 + 粗糙度漸變凹凸）與雙線性 gz。"""
import numpy as np
import mujoco

# --- 幾何參數（見 spec §3）---
PLATFORM_HALF = 1.0
TERR_X_MAX = 7.0
TERR_WY = 3.0
AMP_MAX = 0.12
# 斜坡：5→10→15→20→25→30°，每段水平 1m，往外漸陡（上/下坡對稱）
_ANG = np.array([5.0, 10.0, 15.0, 20.0, 25.0, 30.0])
_SEG = 1.0
_rise = np.cumsum(_SEG * np.tan(np.radians(_ANG)))     # 累積高度 at x=2..7
_zpos = np.concatenate([[0.0], _rise])                 # at x=1..7（+x 上坡）
KNOTS_X = np.array([-7.0, -6.0, -5.0, -4.0, -3.0, -2.0, -1.0,
                    1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0], np.float64)
KNOTS_Z = np.concatenate([-_zpos[::-1], _zpos])        # -x 下坡對稱負值


def slope_z(x):
    return np.interp(x, KNOTS_X, KNOTS_Z)


def amp_at(x):
    return AMP_MAX * np.clip((np.abs(x) - PLATFORM_HALF) / 2.0, 0.0, 1.0)


def bump(x, y):
    # 多正弦疊加，正規化到 ~[-1,1]；確定性（幾何靜態）
    s = (np.sin(2.1 * x) * np.cos(1.7 * y)
         + 0.5 * np.sin(3.7 * x + 1.0) * np.cos(2.9 * y + 0.5)
         + 0.3 * np.sin(5.3 * x + 2.0) * np.cos(4.1 * y))
    return s / 1.8


def build_height_grid(ncol=189, nrow=81):
    xs = np.linspace(-TERR_X_MAX, TERR_X_MAX, ncol)
    ys = np.linspace(-TERR_WY, TERR_WY, nrow)
    X, Y = np.meshgrid(xs, ys)                      # (nrow, ncol)
    Hg = slope_z(X) + amp_at(X) * bump(X, Y)
    return xs, ys, Hg


XS, YS, H = build_height_grid()


def gz_from(xp, xs, ys, Hg, x, y):
    """array-agnostic 雙線性內插；xp = numpy 或 jax.numpy。均勻網格→直接算索引。"""
    nx = xs.shape[0]; ny = ys.shape[0]
    fx = (x - xs[0]) / (xs[-1] - xs[0]) * (nx - 1)
    fy = (y - ys[0]) / (ys[-1] - ys[0]) * (ny - 1)
    fx = xp.clip(fx, 0.0, nx - 1 - 1e-6)
    fy = xp.clip(fy, 0.0, ny - 1 - 1e-6)
    ix = xp.floor(fx).astype(xp.int32); iy = xp.floor(fy).astype(xp.int32)
    tx = fx - ix; ty = fy - iy
    h00 = Hg[iy, ix]; h01 = Hg[iy, ix + 1]
    h10 = Hg[iy + 1, ix]; h11 = Hg[iy + 1, ix + 1]
    return (h00 * (1 - tx) * (1 - ty) + h01 * tx * (1 - ty)
            + h10 * (1 - tx) * ty + h11 * tx * ty)


def gz_np(x, y):
    return gz_from(np, XS, YS, H, np.asarray(x, np.float64), np.asarray(y, np.float64))


def build_terrain3_model(scene_path):
    spec = mujoco.MjSpec.from_file(scene_path)
    floor = next(g for g in spec.geoms if g.name == "floor")
    hmin = float(H.min()); hmax = float(H.max())
    data01 = ((H - hmin) / (hmax - hmin)).astype(np.float64)   # [0,1] row-major
    hf = spec.add_hfield()
    hf.name = "terrain3"
    hf.nrow = H.shape[0]; hf.ncol = H.shape[1]
    hf.size = [TERR_X_MAX, TERR_WY, (hmax - hmin), 0.5]
    hf.userdata = data01.flatten().tolist()
    floor.type = mujoco.mjtGeom.mjGEOM_HFIELD
    floor.hfieldname = "terrain3"
    floor.pos = [0.0, 0.0, hmin]                # data=0(最低) 對到世界 z=hmin → 平台(H=0) 落在 z=0
    # 安全底網：加一塊大 plane 在 z=-10
    net = spec.worldbody.add_geom()
    net.name = "safety_net"; net.type = mujoco.mjtGeom.mjGEOM_PLANE
    net.size = [0.0, 0.0, 0.05]; net.pos = [0.0, 0.0, -10.0]
    net.rgba = [0.3, 0.3, 0.3, 0.0]
    return spec.compile()

In [ ]:
%%writefile obs3.py
# === 模組: obs3.py（v3：改自 v2，Colab 內自成一體）===
"""觀測建構：欄位順序與 v1 一致，last_action 長度決定 76/80。"""
import jax.numpy as jnp


def build_obs(grav, blin, gyro, dq, dqvel, cmd, last_action, contact, c):
    return jnp.concatenate([
        grav, blin, gyro,
        dq, dqvel,
        cmd, last_action, contact,
        c["rx"], c["rx_d"], c["ry"], c["ry_d"],
        jnp.sin(c["theta"]), jnp.cos(c["theta"]),
    ])

In [ ]:
%%writefile go2_terrain3b_env.py
# === 模組: go2_terrain3b_env.py（v3：改自 v2，Colab 內自成一體）===
# 注意：__init__ 內 zeroes geom_margin/geom_gap 是 MJX hfield-sphere 碰撞必要修正，勿刪。
"""Go2 CPG-RL 地形 v3b：terrain3 + slip懲罰減弱(0.1) + 爬坡獎勵，回復爬坡力。"""
import numpy as np
import mujoco
from mujoco import mjx
import jax
import jax.numpy as jnp
from brax.envs.base import Env, State

import terrain3 as T
import cpg3 as C
import obs3 as O

SCENE_MJX = "mujoco_menagerie/unitree_go2/scene_mjx.xml"
CTRL_DT, SIM_DT = 0.02, 0.004
N_FRAMES = int(round(CTRL_DT / SIM_DT))
HOME12 = jnp.array([0.0, 0.9, -1.8] * 4)
KP_NOM, KD_NOM = 90.0, 3.0
KNEE_IDX = [2, 5, 8, 11]
FOOT_CONTACT_H = 0.03
PUSH_EVERY = 100
PUSH_VEL = 0.6
W_SLIP = 0.1                       # v3b：0.5→0.1（原值過度保守、害爬坡）
# terrain 網格常數轉 jnp（gz 用）
XS_J = jnp.asarray(T.XS); YS_J = jnp.asarray(T.YS); H_J = jnp.asarray(T.H)


def gz_j(x, y):
    return T.gz_from(jnp, XS_J, YS_J, H_J, x, y)


def apply_pd(m, kp=KP_NOM, kd=KD_NOM):
    m.actuator_gainprm[:, 0] = kp
    m.actuator_biasprm[:, 0] = 0.0
    m.actuator_biasprm[:, 1] = -kp
    m.actuator_biasprm[:, 2] = -kd
    fr = np.full(m.nu, 23.7); fr[KNEE_IDX] = 45.43
    m.actuator_forcerange[:, 0] = -fr; m.actuator_forcerange[:, 1] = fr
    m.actuator_forcelimited[:] = 1
    return m


def _qinv(q): return jnp.array([q[0], -q[1], -q[2], -q[3]])
def _qrot(q, v):
    u = q[1:4]; t = 2.0 * jnp.cross(u, v); return v + q[0] * t + jnp.cross(u, t)
def w2b(quat, v): return _qrot(_qinv(quat), v)


class Go2Terrain3bEnv(Env):
    def __init__(self, jinvs):
        m = T.build_terrain3_model(SCENE_MJX); m.opt.timestep = SIM_DT
        m = apply_pd(m)
        # go2_mjx.xml 的 default geom 設 margin=0.001，MJX 的 hfield-sphere
        # 碰撞未實作 margin/gap（put_model 會 raise）→ 清零讓 hfield 足端碰撞可用
        m.geom_margin[:] = 0.0
        m.geom_gap[:] = 0.0
        self._mj = m
        self.sys = mjx.put_model(m)
        self._init_q = jnp.array(m.key_qpos[0])
        self._lo = jnp.array(m.actuator_ctrlrange[:, 0])
        self._hi = jnp.array(m.actuator_ctrlrange[:, 1])
        self._jinvs = jnp.array(jinvs)
        self._foot_gid = jnp.array(
            [mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, lg) for lg in C.LEGS])

    @property
    def observation_size(self): return 80
    @property
    def action_size(self): return 16
    @property
    def backend(self): return "mjx"

    def _sample_cmd(self, rng):
        k1, k2, k3 = jax.random.split(rng, 3)
        vx = jax.random.uniform(k1, (), minval=0.0, maxval=1.0)
        vy = jax.random.uniform(k2, (), minval=-0.3, maxval=0.3)
        wz = jax.random.uniform(k3, (), minval=-1.0, maxval=1.0)
        return jnp.array([vx, vy, wz])

    def _base(self, data):
        quat = data.qpos[3:7]; gyro = data.qvel[3:6]
        blin = w2b(quat, data.qvel[0:3])
        grav = w2b(quat, jnp.array([0.0, 0.0, -1.0]))
        return quat, gyro, blin, grav

    def _foot_contact(self, data):
        fx = data.geom_xpos[self._foot_gid, 0]
        fy = data.geom_xpos[self._foot_gid, 1]
        fz = data.geom_xpos[self._foot_gid, 2]
        return (fz - gz_j(fx, fy) < FOOT_CONTACT_H).astype(jnp.float32)

    def _obs(self, data, info):
        _, gyro, blin, grav = self._base(data)
        obs = O.build_obs(grav, blin, gyro,
                          data.qpos[7:19] - HOME12, data.qvel[6:18],
                          info["cmd"], info["last_action"],
                          self._foot_contact(data), info["cpg"])
        return jnp.nan_to_num(jnp.clip(obs, -50.0, 50.0), nan=0.0)

    def reset(self, rng):
        rng, crng, hrng = jax.random.split(rng, 3)
        downhill = jax.random.bernoulli(hrng, 0.5)
        quat = jnp.where(downhill, jnp.array([0.0, 0.0, 0.0, 1.0]),
                         jnp.array([1.0, 0.0, 0.0, 0.0]))
        qpos = self._init_q.at[3:7].set(quat)
        data = mjx.make_data(self.sys).replace(qpos=qpos)
        data = mjx.forward(self.sys, data)
        info = {"rng": rng, "cmd": self._sample_cmd(crng),
                "cpg": C.cpg_init(), "last_action": jnp.zeros(16),
                "step": jnp.zeros((), jnp.int32),
                "foot_xy": data.geom_xpos[self._foot_gid, :2]}
        obs = self._obs(data, info)
        metrics = {"reward": jnp.zeros(()), "r_lin": jnp.zeros(()),
                   "r_yaw": jnp.zeros(()), "rel_h": jnp.zeros(()),
                   "gc_mean": jnp.zeros(()), "scuff": jnp.zeros(()),
                   "slip": jnp.zeros(())}
        return State(data, obs, jnp.zeros(()), jnp.zeros(()), metrics, info)

    def step(self, state, action):
        mux, muy, omega, gc = C.action_to_cpg_cmd(action, "learnable")
        cpg = C.cpg_step(state.info["cpg"], mux, muy, omega, CTRL_DT)
        q_des = C.cpg_to_joint_targets(cpg, self._jinvs, gc)
        ctrl = jnp.clip(q_des, self._lo, self._hi)

        def one(d, _):
            return mjx.step(self.sys, d.replace(ctrl=ctrl)), None
        data, _ = jax.lax.scan(one, state.pipeline_state, None, N_FRAMES)

        rng, krng = jax.random.split(state.info["rng"])
        step_i = state.info["step"] + 1
        do_push = jnp.mod(step_i, PUSH_EVERY) == 0
        kick = jax.random.uniform(krng, (2,), minval=-PUSH_VEL, maxval=PUSH_VEL)
        qvel = (data.qvel.at[0].add(jnp.where(do_push, kick[0], 0.0))
                          .at[1].add(jnp.where(do_push, kick[1], 0.0)))
        data = data.replace(qvel=qvel)
        foot_xy = data.geom_xpos[self._foot_gid, :2]

        info = {**state.info, "cpg": cpg, "last_action": action,
                "rng": rng, "step": step_i, "foot_xy": foot_xy}
        obs = self._obs(data, info)
        _, gyro, blin, grav = self._base(data)
        cmd = info["cmd"]
        r_lin = jnp.exp(-((blin[0] - cmd[0]) ** 2 + (blin[1] - cmd[1]) ** 2) / 0.25)
        r_yaw = jnp.exp(-((gyro[2] - cmd[2]) ** 2) / 0.25)
        upright = grav[0] ** 2 + grav[1] ** 2
        gzb = gz_j(data.qpos[0], data.qpos[1])
        rel_h = data.qpos[2] - gzb
        # rel_h 夾在 [0,0.6]：機身被彈飛也不讓 height_pen 無上界爆炸(≤0.09)
        height_pen = (jnp.clip(rel_h, 0.0, 0.6) - 0.30) ** 2
        # act_rate 對「tanh 後的有界動作」算：raw 輸出飽和後行為不變，
        # 若仍用 raw 差分，一旦 policy 發散懲罰會隨原始輸出無界爆炸(曾致 -1e5 reward)。
        a_sq = jnp.tanh(action)
        la_sq = jnp.tanh(state.info["last_action"])
        act_rate = jnp.sum((a_sq - la_sq) ** 2)
        # 擺動卡住懲罰：擺動期(sinθ>0)的腳若仍觸地(=卡到凸起/拖地)就扣，sinθ 加權。
        # 因地制宜：平地擺動腳不觸地→不罰；凹凸地擺動腳撞凸起→罰→逼它把 gc 抬高跨過。
        swing = jnp.clip(jnp.sin(cpg["theta"]), 0.0, None)
        contact = self._foot_contact(data)
        # 滑動懲罰：觸地支撐腳的世界水平滑動速度（planted 腳理應原地不動；有界 [0,4]）
        slip_vel = jnp.linalg.norm(foot_xy - state.info["foot_xy"], axis=1) / CTRL_DT
        slip = jnp.sum(contact * jnp.clip(slip_vel, 0.0, 1.0))
        scuff = jnp.sum(contact * swing)       # 有界 [0,4]
        r_prog = jnp.clip(blin[0], 0.0, cmd[0])                  # 前進/爬坡獎勵(不飽和,逼它別在坡上停住)
        reward = (1.5 * r_lin + 1.2 * r_yaw + 0.6 * r_prog - 1.0 * upright
                  - 0.5 * height_pen - 0.05 * act_rate
                  - 0.4 * scuff - W_SLIP * slip + 0.05)                          # 無 y_pen
        # 保險下界：正常單步 reward∈~[-2,2.75]，-5 只截極端病態步、不影響正常學習
        reward = jnp.maximum(reward, -5.0)
        done = jnp.where((rel_h < 0.18) | (grav[2] > -0.4), 1.0, 0.0)
        finite = (jnp.isfinite(reward) & jnp.all(jnp.isfinite(data.qpos))
                  & jnp.all(jnp.isfinite(data.qvel)))
        reward = jnp.where(finite, reward, 0.0)
        done = jnp.where(finite, done, 1.0)
        metrics = {"reward": reward, "r_lin": r_lin, "r_yaw": r_yaw,
                   "rel_h": rel_h, "gc_mean": jnp.mean(gc), "scuff": scuff,
                   "slip": slip}
        metrics = {k: jnp.nan_to_num(v) for k, v in metrics.items()}  # 防某env爆掉污染監控
        return state.replace(pipeline_state=data, obs=obs, reward=reward,
                             done=done, metrics=metrics, info=info)


_mm = T.build_terrain3_model(SCENE_MJX)
BASE_ID = mujoco.mj_name2id(_mm, mujoco.mjtObj.mjOBJ_BODY, "base")


def domain_randomize(sys, rng):
    @jax.vmap
    def per_env(rng):
        k1, k2, k3, k4, k5 = jax.random.split(rng, 5)
        geom_friction = sys.geom_friction.at[:, 0].set(
            jax.random.uniform(k1, minval=0.5, maxval=1.25))
        kp = jax.random.uniform(k2, minval=75.0, maxval=105.0)
        kd = jax.random.uniform(k3, minval=2.0, maxval=4.0)
        gain = sys.actuator_gainprm.at[:, 0].set(kp)
        bias = sys.actuator_biasprm.at[:, 1].set(-kp).at[:, 2].set(-kd)
        body_mass = sys.body_mass * jax.random.uniform(
            k4, (sys.nbody,), minval=0.8, maxval=1.2)
        payload = jax.random.uniform(k5, minval=0.0, maxval=8.0)
        body_mass = body_mass.at[BASE_ID].add(payload)
        return geom_friction, gain, bias, body_mass
    gf, gain, bias, bm = per_env(rng)
    in_axes = jax.tree_util.tree_map(lambda x: None, sys)
    in_axes = in_axes.replace(geom_friction=0, actuator_gainprm=0,
                              actuator_biasprm=0, body_mass=0)
    sys = sys.replace(geom_friction=gf, actuator_gainprm=gain,
                      actuator_biasprm=bias, body_mass=bm)
    return sys, in_axes


import 四個模組（Colab 單一 namespace；env 內部以 `import terrain3 as T` 等引用，故需先把檔案寫出再 import，與模組零差異、免改前綴）。

In [ ]:
import importlib
import cpg3, terrain3, obs3
import go2_terrain3b_env
importlib.reload(cpg3); importlib.reload(terrain3)
importlib.reload(obs3); importlib.reload(go2_terrain3b_env)
from go2_terrain3b_env import Go2Terrain3bEnv, domain_randomize, apply_pd
import cpg3 as C, terrain3 as T, obs3 as O
print("modules imported: cpg3 / terrain3 / obs3 / go2_terrain3b_env")

## 第 3 步：Smoke test（開訓練前必跑）
檢查 obs=80、上/下坡 spawn、reward 有限、done 不誤觸發、`gc_mean` 有值。

In [ ]:
import cpg3 as C, go2_terrain3b_env as E
import jax, jax.numpy as jnp
jinvs = C.leg_ik_consts("mujoco_menagerie/unitree_go2/scene_mjx.xml")
env = E.Go2Terrain3bEnv(jinvs)
for seed in [0, 1, 2, 3]:
    s = jax.jit(env.reset)(jax.random.PRNGKey(seed))
    s2 = jax.jit(env.step)(s, jnp.zeros(16))
    face = "下坡(-x)" if float(s.pipeline_state.qpos[6]) > 0.5 else "上坡(+x)"
    print(f"[seed {seed}] obs={s.obs.shape} {face} reward={float(s2.reward):+.3f} "
          f"done={float(s2.done):.0f} rel_h={float(s2.metrics['rel_h']):.3f} "
          f"gc_mean={float(s2.metrics['gc_mean']):.3f} "
          f"slip={float(s2.metrics['slip']):.3f}")
print("PASSED" if s.obs.shape == (80,) else "CHECK OBS SIZE")

## 第 4 步：Brax PPO 訓練
v3b：slip 懲罰減弱 + 爬坡獎勵，其餘同 v3。`num_timesteps=3e8`。OOM 就降 `num_envs`。
`environment=Go2Terrain3bEnv(jinvs)`、`randomization_fn=domain_randomize`。

In [ ]:
import functools, time
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

env = Go2Terrain3bEnv(jinvs)
network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=(256, 256, 128),
    value_hidden_layer_sizes=(256, 256, 256))

train_fn = functools.partial(
    ppo.train, num_timesteps=300_000_000, num_evals=20, episode_length=1000,
    num_envs=2048, batch_size=256, num_minibatches=32, unroll_length=20,
    num_updates_per_batch=4, learning_rate=3e-4, entropy_cost=3e-3,
    discounting=0.97, normalize_observations=True,
    network_factory=network_factory, randomization_fn=domain_randomize, seed=0)

_t0 = time.time(); rewards = []
def progress(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0)); rewards.append((step, r))
    # brax 把 env 的 metrics 聚合成 eval/episode_<名字>(整段 episode 加總)，除以長度得每步平均
    L = float(metrics.get("eval/avg_episode_length", 0.0)) or 1.0
    gc = float(metrics.get("eval/episode_gc_mean", 0.0)) / L      # 每步平均抬腳 gc(m)
    scuff = float(metrics.get("eval/episode_scuff", 0.0)) / L     # 每步平均擺動卡住量(越小越順)
    relh = float(metrics.get("eval/episode_rel_h", 0.0)) / L      # 每步平均機身相對地面高(m)
    slip = float(metrics.get("eval/episode_slip", 0.0)) / L      # 每步平均滑動量(越小越不打滑)
    print(f"step {step:>11,}  reward {r:8.1f}  len {L:5.0f}  "
          f"gc {gc:.3f}  scuff {scuff:.3f}  slip {slip:.3f}  relh {relh:.2f}  ({time.time()-_t0:.0f}s)")

make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)
print("training done")

In [ ]:
import matplotlib.pyplot as plt
plt.plot([s for s, _ in rewards], [r for _, r in rewards], marker="o")
plt.xlabel("env steps"); plt.ylabel("eval reward"); plt.grid(True); plt.show()

## 第 5 步：Rollout 影片（展示全向指令跟隨）
CPU numpy 迴圈，obs 組法呼叫嵌入的 `obs3.build_obs`、CPG 呼叫 `cpg3`、地面用 `terrain3.gz_np`，與訓練逐項一致（含 gz 相對地面觸地）。跑三段各出一支影片：
- **直走上坡**：`cmd=[0.6, 0, 0]`、spawn 面 +x；
- **轉向**：`cmd=[0.4, 0, 0.8]`（邊走邊左轉）；
- **橫移**：`cmd=[0.3, 0.25, 0]`（前進兼右移）。
每段印 `前進 / 末端高 / FL抬腳量 / gc_mean`。

In [ ]:
import numpy as np, mujoco, mediapy as media
import jax, jax.numpy as jnp

infer = jax.jit(make_inference_fn(params, deterministic=True))
HOME12 = np.array([0.0, 0.9, -1.8] * 4)
CTRL_DT, SIM_DT = 0.02, 0.004
JINVS_np = C.leg_ik_consts(SCENE)


def _qinv(q): return np.array([q[0], -q[1], -q[2], -q[3]])
def _qrot(q, v):
    u = q[1:4]; t = 2 * np.cross(u, v); return v + q[0] * t + np.cross(u, t)
def w2b(q, v): return _qrot(_qinv(q), v)


def _make_model():
    m = T.build_terrain3_model(SCENE); m.opt.timestep = SIM_DT
    m = apply_pd(m)
    return m


def rollout(cmd, downhill=False, secs=8.0, title=""):
    m = _make_model()
    d = mujoco.MjData(m); mujoco.mj_resetDataKeyframe(m, d, 0)
    d.qpos[3:7] = [0, 0, 0, 1.0] if downhill else [1.0, 0, 0, 0]   # 面 -x / +x
    mujoco.mj_forward(m, d)
    lo = m.actuator_ctrlrange[:, 0]; hi = m.actuator_ctrlrange[:, 1]
    foot_gid = [mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, lg) for lg in C.LEGS]
    n_sub = int(round(CTRL_DT / SIM_DT))
    c = C.cpg_init(); last_a = np.zeros(16); cmd = np.asarray(cmd, np.float32)
    ren = mujoco.Renderer(m, 480, 640); cam = mujoco.MjvCamera()
    mujoco.mjv_defaultFreeCamera(m, cam)
    frames = []; x0 = float(d.qpos[0]); fl = foot_gid[0]
    fzmin = fzmax = float(d.geom_xpos[fl][2]); gc_hist = []
    for i in range(int(secs / CTRL_DT)):
        grav = w2b(d.qpos[3:7], np.array([0, 0, -1.0]))
        blin = w2b(d.qpos[3:7], d.qvel[0:3])
        fx = np.array([d.geom_xpos[g][0] for g in foot_gid])
        fy = np.array([d.geom_xpos[g][1] for g in foot_gid])
        fz = np.array([d.geom_xpos[g][2] for g in foot_gid])
        gzf = np.array([float(T.gz_np(fx[k], fy[k])) for k in range(4)])
        contact = ((fz - gzf) < 0.03).astype(np.float32)
        o = O.build_obs(jnp.asarray(grav), jnp.asarray(blin), jnp.asarray(d.qvel[3:6]),
                        jnp.asarray(d.qpos[7:19] - HOME12), jnp.asarray(d.qvel[6:18]),
                        jnp.asarray(cmd), jnp.asarray(last_a), jnp.asarray(contact), c)
        act = np.array(infer(jnp.asarray(o, jnp.float32), jax.random.PRNGKey(0)))
        mux, muy, om, gc = C.action_to_cpg_cmd(jnp.asarray(act), "learnable")
        gc_hist.append(float(np.mean(np.asarray(gc))))
        c = C.cpg_step(c, mux, muy, om, CTRL_DT)
        q_des = np.array(C.cpg_to_joint_targets(c, jnp.asarray(JINVS_np), gc))
        d.ctrl[:] = np.clip(q_des, lo, hi)
        for _ in range(n_sub):
            mujoco.mj_step(m, d)
        last_a = act
        flz = float(d.geom_xpos[fl][2])
        fzmin = min(fzmin, flz); fzmax = max(fzmax, flz)
        if i % 2 == 0:
            cam.lookat[:] = [d.qpos[0], d.qpos[1], 0.3]; cam.distance = 2.5
            cam.elevation = -18; cam.azimuth = 90
            ren.update_scene(d, cam); frames.append(ren.render())
    dist = float(d.qpos[0]) - x0
    end_gz = float(T.gz_np(d.qpos[0], d.qpos[1]))
    print(f"[{title}] 前進={dist:+.2f}m 末端地面高={end_gz:+.2f}m "
          f"末端rel_h={float(d.qpos[2]) - end_gz:.2f}m FL抬腳量={fzmax - fzmin:.3f}m "
          f"gc_mean={np.mean(gc_hist):.3f}")
    return frames


print("=== 直走上坡 cmd=[0.6,0,0] ===")
f_straight = rollout([0.6, 0.0, 0.0], downhill=False, title="直走上坡")
print("=== 轉向 cmd=[0.4,0,0.8] ===")
f_turn = rollout([0.4, 0.0, 0.8], downhill=False, title="轉向")
print("=== 橫移 cmd=[0.3,0.25,0] ===")
f_strafe = rollout([0.3, 0.25, 0.0], downhill=False, title="橫移")
media.show_video(f_straight, fps=25)
media.show_video(f_turn, fps=25)
media.show_video(f_strafe, fps=25)

## 第 6 步：存權重下載
存成 `cpg_rl_terrain3b_params.pkl`（16 維 learnable 權重）。帶回本機放 `task4/weights/`，用 `local_infer_terrain3.py` 出對比影片。

In [ ]:
from brax.io import model
model.save_params("cpg_rl_terrain3b_params.pkl", params)
try:
    from google.colab import files; files.download("cpg_rl_terrain3b_params.pkl")
except Exception as e:
    print("左側檔案面板右鍵下載 cpg_rl_terrain3b_params.pkl。", e)